# 图片文件夹数据集索引与划分

这个 notebook 用于处理已经按类别整理好的图片文件夹，自动生成 `train.txt`、`val.txt` 索引文件，并提供一个基于 `torch.utils.data.Dataset` 的子类来读取这些索引。数据集按 9:1 的比例划分，索引文件以制表符分隔保存。

## 1. 导入依赖与基础配置

导入基础库、定义图片后缀、默认输入目录和输出目录。

In [22]:
from __future__ import annotations

import logging

import click
import random
from pathlib import Path
from typing import Iterable

from PIL import Image
from torch.utils.data import Dataset

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DEFAULT_ROOT_DIR = Path(r"d:\Jupyter Lab\CiFar10\cifar-10-batches-py\export")
DEFAULT_OUTPUT_DIR = Path(r"d:\Jupyter Lab\CiFar10\dataset_index")
DEFAULT_LOG_DIR = Path(r"d:\Jupyter Lab\CiFar10\Log")
DEFAULT_LOG_NAME = "image_folder_dataset_index.log"
DEFAULT_SEED = 42


def setup_logging(log_dir: Path = DEFAULT_LOG_DIR, log_name: str = DEFAULT_LOG_NAME) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("image_folder_dataset_index")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()

    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    file_handler = logging.FileHandler(log_dir / log_name, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    return logger

## 2. 扫描图片并生成索引

按类别文件夹扫描图片，打乱后按 9:1 划分，并把每个样本写成一行 `相对路径\t类别id`。

In [23]:
def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def collect_class_folders(root_dir: Path) -> list[Path]:
    return sorted([path for path in root_dir.iterdir() if path.is_dir()])


def collect_samples(root_dir: Path) -> list[tuple[Path, int, str]]:
    class_folders = collect_class_folders(root_dir)
    samples: list[tuple[Path, int, str]] = []

    for class_index, class_folder in enumerate(class_folders):
        for image_path in sorted(class_folder.rglob("*")):
            if is_image_file(image_path):
                relative_path = image_path.relative_to(root_dir).as_posix()
                samples.append((image_path, class_index, relative_path))

    return samples


def split_class_samples(samples: list[tuple[Path, int, str]], seed: int = DEFAULT_SEED) -> tuple[list[tuple[Path, int, str]], list[tuple[Path, int, str]]]:
    rng = random.Random(seed)
    grouped_samples: dict[int, list[tuple[Path, int, str]]] = {}
    for sample in samples:
        grouped_samples.setdefault(sample[1], []).append(sample)

    train_samples: list[tuple[Path, int, str]] = []
    val_samples: list[tuple[Path, int, str]] = []

    for class_index in sorted(grouped_samples):
        class_samples = grouped_samples[class_index][:]
        rng.shuffle(class_samples)
        total = len(class_samples)
        train_count = int(total * 0.9)
        val_count = total - train_count

        train_samples.extend(class_samples[:train_count])
        val_samples.extend(class_samples[train_count:train_count + val_count])

    return train_samples, val_samples


def write_index_file(index_path: Path, samples: Iterable[tuple[Path, int, str]]) -> int:
    index_path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with index_path.open("w", encoding="utf-8") as handle:
        for _, class_index, relative_path in samples:
            handle.write(f"{relative_path}\t{class_index}\n")
            count += 1
    return count


def build_and_save_indices(root_dir: Path = DEFAULT_ROOT_DIR, output_dir: Path = DEFAULT_OUTPUT_DIR, seed: int = DEFAULT_SEED, logger: logging.Logger | None = None) -> dict[str, int]:
    root_dir = root_dir.resolve()
    output_dir = output_dir.resolve()
    logger = logger or logging.getLogger("image_folder_dataset_index")
    logger.info("开始生成索引文件: root_dir=%s output_dir=%s seed=%s", root_dir, output_dir, seed)
    samples = collect_samples(root_dir)
    logger.info("扫描到图片总数: %s", len(samples))
    train_samples, val_samples = split_class_samples(samples, seed=seed)
    logger.info("划分结果: train=%s val=%s", len(train_samples), len(val_samples))

    test_index = output_dir / "test.txt"
    if test_index.exists():
        test_index.unlink()
        logger.info("已删除旧的 test 索引文件: %s", test_index)

    counts = {
        "train": write_index_file(output_dir / "train.txt", train_samples),
        "val": write_index_file(output_dir / "val.txt", val_samples),
    }
    logger.info("索引文件写入完成: %s", counts)
    return counts

## 3. 基于 Dataset 的数据集子类

读取索引 txt，按需加载图片并返回图像与标签。

In [24]:
class ImageIndexDataset(Dataset):
    def __init__(self, index_file: Path, root_dir: Path | None = None, transform=None, target_transform=None):
        self.index_file = Path(index_file)
        self.root_dir = Path(root_dir) if root_dir is not None else self.index_file.parent
        self.transform = transform
        self.target_transform = target_transform
        self.samples: list[tuple[Path, int]] = []

        with self.index_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue
                relative_path, label_text = line.split("\t")
                image_path = Path(relative_path)
                if not image_path.is_absolute():
                    image_path = self.root_dir / image_path
                self.samples.append((image_path, int(label_text)))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            label = self.target_transform(label)

        return image, label


## 4. 生成索引并验证数据集

先生成 `train.txt`、`val.txt`，再用 `ImageIndexDataset` 验证读取结果。

In [25]:
# 你可以把 root_dir 改成自己的图片根目录，目录结构应当类似：
# root_dir/
#   class_a/*.jpg
#   class_b/*.jpg
root_dir = DEFAULT_ROOT_DIR
output_dir = DEFAULT_OUTPUT_DIR
logger = setup_logging()

counts = build_and_save_indices(root_dir=root_dir, output_dir=output_dir, seed=DEFAULT_SEED, logger=logger)
train_dataset = ImageIndexDataset(output_dir / "train.txt", root_dir=root_dir)
val_dataset = ImageIndexDataset(output_dir / "val.txt", root_dir=root_dir)

logger.info("索引文件已生成: %s", counts)
logger.info("数据集长度: %s", {
    "train": len(train_dataset),
    "val": len(val_dataset),
})

if len(train_dataset) > 0:
    sample_image, sample_label = train_dataset[0]
    logger.info("训练集首个样本标签: %s", sample_label)
    logger.info("训练集首个样本尺寸: %s", sample_image.size if hasattr(sample_image, "size") else type(sample_image))

log_file = DEFAULT_LOG_DIR / DEFAULT_LOG_NAME
logger.info("日志文件: %s", log_file)
logger.info("日志文件是否存在: %s", log_file.exists())

2026-05-22 15:21:44,891 INFO 开始生成索引文件: root_dir=D:\Jupyter Lab\CiFar10\cifar-10-batches-py\export output_dir=D:\Jupyter Lab\CiFar10\dataset_index seed=42
2026-05-22 15:21:49,147 INFO 扫描到图片总数: 50000
2026-05-22 15:21:49,161 INFO 划分结果: train=45000 val=5000
2026-05-22 15:21:49,190 INFO 索引文件写入完成: {'train': 45000, 'val': 5000}
2026-05-22 15:21:49,615 INFO 索引文件已生成: {'train': 45000, 'val': 5000}
2026-05-22 15:21:49,615 INFO 数据集长度: {'train': 45000, 'val': 5000}
2026-05-22 15:21:49,615 INFO 训练集首个样本标签: 0
2026-05-22 15:21:49,623 INFO 训练集首个样本尺寸: (32, 32)
2026-05-22 15:21:49,624 INFO 日志文件: d:\Jupyter Lab\CiFar10\Log\image_folder_dataset_index.log
2026-05-22 15:21:49,625 INFO 日志文件是否存在: True


## 5. 使用 Click 定义命令行入口

使用 `click.command` 和 `click.option` 接收输入目录、输出目录、随机种子和日志目录参数。

In [26]:
@click.command()
@click.option("--root-dir", type=click.Path(path_type=Path, exists=True, file_okay=False, dir_okay=True), default=DEFAULT_ROOT_DIR, show_default=True, help="图片根目录，目录下按类别存放图片。")
@click.option("--output-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_OUTPUT_DIR, show_default=True, help="索引文件输出目录。")
@click.option("--seed", type=int, default=DEFAULT_SEED, show_default=True, help="随机划分种子。")
@click.option("--log-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_LOG_DIR, show_default=True, help="日志文件保存目录。")
@click.option("--log-name", type=str, default=DEFAULT_LOG_NAME, show_default=True, help="日志文件名。")
def main(root_dir: Path, output_dir: Path, seed: int, log_dir: Path, log_name: str) -> None:
    logger = setup_logging(log_dir=log_dir, log_name=log_name)
    counts = build_and_save_indices(root_dir=root_dir, output_dir=output_dir, seed=seed, logger=logger)
    logger.info("索引生成完成: %s", counts)
    logger.info("日志文件: %s", log_dir / log_name)